# 100 — Persist export / pause（tier_a + `/storage/ssh` → download → purge）

一時停止用。既定は **tier_a**（復元困難分 ≈2 GiB）+ **`/storage/ssh`**。

Zip レイアウト（復元: `cd /storage && unzip -o ARCHIVE.zip`）:
- `validated_4d_su2_rg/...`
- `storage/ssh/...`

**安全弁**
- 既定 dry-run / `EXECUTE=True` で zip
- purge は persist のみ（**`/storage/ssh` は消さない**）
- tier_a purge は `DISCARD_M3_CHECKPOINTS` 確認必須

詳細: `docs/campaign_b_persist_export_pause.md`


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'campaign_b' / 'persist_export.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/campaign_b/persist_export.py not found.')
sys.path.insert(0, str(PROJECT_ROOT))
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
print('PROJECT_ROOT =', PROJECT_ROOT)
print('PERSIST_ROOT =', PERSIST_ROOT)


In [ ]:
# --- knobs ---
EXECUTE = False          # True: write zip
PURGE = False            # True: delete persist AFTER verified zip (never deletes /storage/ssh)
CONFIRM_PURGE = ''       # must be 'PURGE_PERSIST_ROOT' when PURGE
CONFIRM_DISCARD_M3 = ''  # tier_a purge: must be 'DISCARD_M3_CHECKPOINTS'
TIER = 'tier_a'          # 'tier_a' (~2GiB) or 'full' (~57GiB)
INCLUDE_SSH = True       # always pack /storage/ssh into the zip
EXPORT_DIR = Path('/notebooks/persist_exports')  # Jupyter UI download
COMPRESS = False
MARGIN_GIB = 2.0
ALLOW_LIVE_GPU_LEASE = False
ARCHIVE_PATH = None      # optional explicit zip path


In [ ]:
from src.campaign_b.persist_export import (
    DISCARD_M3_CHECKPOINTS_CONFIRM,
    PURGE_CONFIRM,
    export_and_optional_purge,
    plan_export,
    purge_persistent_root,
    verify_archive,
)

plan = plan_export(
    PERSIST_ROOT,
    archive_path=ARCHIVE_PATH,
    export_dir=EXPORT_DIR,
    margin_bytes=int(MARGIN_GIB * 1024**3),
    allow_live_gpu_lease=ALLOW_LIVE_GPU_LEASE,
    tier=TIER,
    include_ssh=INCLUDE_SSH,
)
print(json.dumps(plan.to_dict(), indent=2, ensure_ascii=False, default=str))
if not plan.ok:
    raise RuntimeError(f'BLOCKED: {plan.block_reason}')


In [ ]:
if not EXECUTE:
    print('Dry-run only. Set EXECUTE=True to write the zip.')
    SESSION = {'status': 'DRY_RUN', 'plan': plan.to_dict()}
else:
    SESSION = export_and_optional_purge(
        PERSIST_ROOT,
        archive_path=ARCHIVE_PATH,
        export_dir=EXPORT_DIR,
        execute=True,
        compress=COMPRESS,
        purge=PURGE,
        confirm_purge=CONFIRM_PURGE or None,
        confirm_discard_m3_checkpoints=CONFIRM_DISCARD_M3 or None,
        allow_live_gpu_lease=ALLOW_LIVE_GPU_LEASE,
        margin_bytes=int(MARGIN_GIB * 1024**3),
        tier=TIER,
        include_ssh=INCLUDE_SSH,
        progress_cb=print,
    )
print(json.dumps({
    'status': SESSION.get('status'),
    'tier': (SESSION.get('result') or {}).get('tier') or TIER,
    'archive_path': (SESSION.get('result') or {}).get('archive_path'),
    'sha256': (SESSION.get('result') or {}).get('sha256'),
    'archive_bytes_human': (SESSION.get('result') or {}).get('archive_bytes_human'),
    'extra_roots': (SESSION.get('plan') or {}).get('extra_roots'),
    'purged': (SESSION.get('result') or {}).get('purged'),
}, indent=2, ensure_ascii=False))
if SESSION.get('status') == 'BLOCKED':
    raise RuntimeError(SESSION.get('plan', {}).get('block_reason'))


## ダウンロード後の purge だけ（再 zip しない）

ローカルで zip を受け取ったあと、下のセルで persist だけ消す。
`ARCHIVE_PATH` と `CONFIRM_PURGE='PURGE_PERSIST_ROOT'` をセットしてから実行。


In [ ]:
PURGE_ONLY = False  # True にしてから実行（/storage/ssh は消さない）
if PURGE_ONLY:
    if ARCHIVE_PATH is None:
        raise RuntimeError('Set ARCHIVE_PATH to the downloaded zip')
    if CONFIRM_PURGE != PURGE_CONFIRM:
        raise RuntimeError(f"CONFIRM_PURGE must be {PURGE_CONFIRM!r}")
    if TIER == 'tier_a' and CONFIRM_DISCARD_M3 != DISCARD_M3_CHECKPOINTS_CONFIRM:
        raise RuntimeError(
            f"CONFIRM_DISCARD_M3 must be {DISCARD_M3_CHECKPOINTS_CONFIRM!r}"
        )
    v = verify_archive(ARCHIVE_PATH)
    print('verified', v)
    report = purge_persistent_root(
        PERSIST_ROOT,
        archive_path=ARCHIVE_PATH,
        confirm_purge=CONFIRM_PURGE,
        execute=True,
        expected_sha256=v['sha256'],
        tier=TIER,
        confirm_discard_m3_checkpoints=CONFIRM_DISCARD_M3 or None,
    )
    print(json.dumps(report, indent=2, ensure_ascii=False, default=str))
else:
    print('Skipped. Set PURGE_ONLY=True after download.')
